In [8]:
import ee 
import geemap

ee.Initialize(project="fall-foliage-project")

UNIV_BUFFER = 40000

locations = { 
    "Raleigh":    ([-78.69649427302112, 35.7664191909458],  "e41a1c"),
    "Boone":      ([-81.68516978460896, 36.21834246106182], "377eb8"),
    "Asheville":  ([-82.55831963941482, 35.633546529345516], "4daf4a"),
    "Charleston": ([-79.9294956284804,  32.796133071541156], "984ea3"),
    "Shenandoah": ([-78.40685152551191, 38.499092552797826], "ff7f00"),

}

regions = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c).buffer(UNIV_BUFFER), {"name": n})
    for n, (c, _) in locations.items()
])

Map = geemap.Map(center=[35.3, -80.0], zoom=6)
Map.add_basemap("HYBRID")

ele = ee.Image("USGS/SRTMGL1_003")
Map.addLayer(ele, {"min": 0, "max": 1800,
                   "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"]},
             "Elevation (SRTM)", False, 0.6)

for name, (coords, color) in locations.items():
    center = ee.Geometry.Point(coords)
    circle = ee.FeatureCollection([ee.Feature(center.buffer(UNIV_BUFFER))])

    
    Map.addLayer(circle.style(color=color, fillColor=color + "40", width=2),
                 {}, f"{name} region")
    Map.addLayer(center, {"color": color}, f"{name} center")


Map.add_legend(title="Study regions",
               legend_dict={n: "#" + col for n, (_, col) in locations.items()})


Map

Map(center=[35.3, -80.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [10]:
import pandas as pd

# ===== Study regions: city circle ∩ the EPA Level III ecoregion it sits in =====
ecoregions = ee.FeatureCollection("EPA/Ecoregions/2013/L3")

region_features = []
for name, (coords, color) in locations.items():
    center = ee.Geometry.Point(coords)
    circle = center.buffer(UNIV_BUFFER)

    home_eco = ee.Feature(ecoregions.filterBounds(center).first())
    clipped = circle.intersection(home_eco.geometry(), ee.ErrorMargin(100))

    region_features.append(ee.Feature(clipped, {
        "name": name,
        "ecoregion": home_eco.get("us_l3name"),
    }))

study_regions = ee.FeatureCollection(region_features)

# ===== Forest mask: NLCD 2021, deciduous (41) + mixed (43) forest =====
nlcd = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021").select("landcover")
forest_mask = nlcd.eq(41).Or(nlcd.eq(43))          # 1 = forest, 0 = everything else

# ===== How much forest is in each region? =====
area_img = ee.Image.pixelArea().divide(1e6)          # km² per pixel
stats_img = (area_img.rename("total_km2")
             .addBands(area_img.updateMask(forest_mask).rename("forest_km2")))

region_stats = stats_img.reduceRegions(
    collection=study_regions,
    reducer=ee.Reducer.sum(),
    scale=30,
    tileScale=4,
).map(lambda f: f.set(
    "forest_frac", ee.Number(f.get("forest_km2")).divide(f.get("total_km2"))
))

# Pull results into pandas (avoids the geemap ee_to_df / pandas bug)
features = region_stats.getInfo()["features"]
df = pd.DataFrame([f["properties"] for f in features])
print(df[["name", "ecoregion", "total_km2", "forest_km2", "forest_frac"]].round(3))

# ===== Display =====
Map.addLayer(forest_mask.selfMask().clip(study_regions),
             {"palette": ["1b7837"]}, "Deciduous + mixed forest")
Map.addLayer(study_regions.style(color="ffffff", fillColor="00000000", width=2),
             {}, "Study regions (ecoregion-clipped)")
Map.centerObject(study_regions, 6)
Map

         name               ecoregion  total_km2  forest_km2  forest_frac
0     Raleigh                Piedmont   4051.605    1052.068        0.260
1       Boone              Blue Ridge   4591.950    3560.974        0.775
2   Asheville              Blue Ridge   4963.414    3474.239        0.700
3  Charleston  Southern Coastal Plain   1649.034       9.841        0.006
4  Shenandoah              Blue Ridge   1216.296    1117.522        0.919


Map(bottom=6776.0, center=[35.78771285214042, -80.8144217567992], controls=(WidgetControl(options=['position',…